# Notebook 05 — Sistema de Predicción Integrado y Rango de Ritmo

**Objetivo:** cerrar la primera versión funcional del sistema de predicción multi-distancia
para 5K, 10K, 21K y 42K con salida en rango de ritmo.

**Capas integradas:**
- **Capa 0** — Verificación de consistencia y selección del PR fuente
- **Capa 1** — Riegel calibrado por segmento (NB01/NB03)
- **Capa 2** — Corrección demográfica (NB04)
- **Capa 3** — Evaluación de viabilidad con dataset Run Club

**Salida principal:** `pace_range_fmt` (ej. `'5:11 – 5:29 min/km'`) + metadatos de confianza.

**Módulo producido:** `src/ml/predictor.py` — importable desde FastAPI y la app.

In [ ]:
import sys
import warnings
from pathlib import Path
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')

NOTEBOOK_DIR = Path().resolve()
ROOT         = NOTEBOOK_DIR.parent.parent
DATA_DIR     = ROOT / 'Datasets running'
FIGS_DIR     = ROOT / 'ml' / 'figures'
FIGS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT))

from src.ml.predictor import (
    predict_race_time_range,
    check_pr_consistency,
    PREFERENCE_ORDER,
    MAE_PENALTY_FACTOR,
    EMPIRICAL_SUPPORT,
)
from src.ml.riegel import CALIBRATED_MAE_MIN

print('src/ml/predictor.py cargado correctamente.')
print('Funciones disponibles: predict_race_time_range, check_pr_consistency')

---
## 1. Capa 0 — Verificación de consistencia de PRs declarados

El formulario de ingreso pide al atleta sus mejores tiempos en cada distancia.
Antes de usarlos como input de Riegel, verificamos que sean mutuamente consistentes.
Un PR inconsistente indica: PR desactualizado, error de reporte, o carrera atípica.

In [ ]:
# ─── Caso 1: PRs consistentes ────────────────────────────────────────────────
perfil_consistente = {
    'pr_5k_sec':  1320,    # 22:00  (5K)
    'pr_10k_sec': 2760,    # 46:00  (10K)
    'pr_21k_sec': 5100,    # 1:25:00 (21K)
    'pr_42k_sec': 10680,   # 2:58:00 (42K)
}

r_ok = check_pr_consistency(perfil_consistente, target_distance='42K')
print(f'Status          : {r_ok["status"]}')
print(f'PR recomendado  : {r_ok["recommended_pr"]}  ({r_ok["recommended_pr_sec"]} seg)')
print(f'Notas           : {r_ok["notes"]}')
print()
print('Checks realizados:')
for c in r_ok['checks']:
    flag = '✅' if c['consistent'] else '❌'
    print(f'  {flag} {c["from"]} → {c["to"]}: '
          f'reportado={c["reported_fmt"]}  '
          f'Riegel={c["riegel_pred_fmt"]}  '
          f'Δ={c["delta_min"]} min  '
          f'(umbral={c["threshold_min"]} min)')

In [ ]:
# ─── Caso 2: PR inconsistente (5K élite + 42K muy lento) ─────────────────────
# Atleta que tuvo un 5K muy rápido hace años y un maratón mediocre recientemente
perfil_inconsistente = {
    'pr_5k_sec':  1080,    # 18:00  (5K muy rápido — posiblemente desactualizado)
    'pr_42k_sec': 16200,   # 4:30:00 (42K lento)
}

r_inc = check_pr_consistency(perfil_inconsistente, target_distance='42K')
print(f'Status          : {r_inc["status"]}')
print(f'PR recomendado  : {r_inc["recommended_pr"]}  ({r_inc["recommended_pr_sec"]} seg)')
print()
print('Diagnóstico:')
for nota in r_inc['notes']:
    print(f'  ⚠️  {nota}')

print()
print('─── INTERPRETACIÓN PARA LA APP ──────────────────────────────────────')
print('Este atleta tiene PRs contradictorios.')
print('El sistema debe preguntar: "¿Cuándo fue ese PR de 5K? ¿Es reciente?"')
print('Si el PR de 5K es de hace varios años, la predicción debe usar el 42K.')
print('Decisión actual: usar el PR de la distancia más cercana al target (42K directo).')

In [ ]:
# ─── Caso 3: Solo un PR disponible ───────────────────────────────────────────
perfil_un_pr = {'pr_21k_sec': 5100}
r_solo = check_pr_consistency(perfil_un_pr, target_distance='42K')
print(f'Status: {r_solo["status"]}')
print(f'PR seleccionado: {r_solo["recommended_pr"]}  ({r_solo["recommended_pr_sec"]} seg)')
print(f'Nota: {r_solo["notes"]}')

print()
print('─── LÓGICA DE LA CAPA 0 ────────────────────────────────────────────')
print('Orden de preferencia por distancia objetivo:')
for target, order in PREFERENCE_ORDER.items():
    print(f'  {target}: {" → ".join(order)}')
print()
print('Regla: preferir el PR de la misma distancia (si existe),')
print('luego el PR del subsiguiente más cercano.')
print('La extrapolación ascendente (corto→largo) es más fiable que la descendente.')

---
## 2. Capas 1+2 — Predicciones por distancia con rango de ritmo

Demostramos el sistema integrado con perfiles de atleta reales.
El rango de ritmo usa el MAE por segmento como ancho de banda.

In [ ]:
# ─── Perfil del atleta de prueba del proyecto ─────────────────────────────────
# PR 21K = 1:25:00 (5100 seg) — atleta del proyecto
perfil_andres = {
    'pr_21k_sec': 5100,
    'pr_10k_sec': 2700,   # 45:00 (supuesto para demostración)
}

print('═' * 65)
print('PREDICCIONES PARA EL ATLETA DEL PROYECTO (PR 21K = 1:25:00)')
print('Edad: 35, Género: M')
print('═' * 65)

for dist in ['5K', '10K', '21K', '42K']:
    r = predict_race_time_range(perfil_andres, dist, age=35, gender='M')
    if 'error' in r:
        print(f'  {dist}: ERROR — {r["error"]}')
        continue
    print(f'\n  {dist} — Fuente PR: {r["source_pr"]} ({r["source_pr_fmt"]})')
    print(f'    Ritmo esperado : {r["pace_range_fmt"]}')
    print(f'    Tiempo central : {r["time_center_fmt"]}')
    print(f'    Rango tiempo   : {r["time_range_fmt"]}')
    print(f'    Segmento       : {r["segment"]}  (exp={r["exponent_used"]})')
    print(f'    MAE efectivo   : ±{r["mae_effective_min"]} min')
    print(f'    Correc. demog. : {r["demo_correction_min"]:+.1f} min (age_group={r["age_group"]}, gender={r["gender"]})')
    print(f'    Confianza      : {r["confidence"]}  — {r["confidence_reason"]}')
    print(f'    Soporte empír. : {r["empirical_support"]}')
    if r['direction_note']:
        print(f'    ⚠️  {r["direction_note"]}')

print('\n' + '═' * 65)

In [ ]:
# ─── Tabla resumen para 3 perfiles típicos ────────────────────────────────────
perfiles = [
    {'nombre': 'Recreativo lento',    'pr_21k_sec': 7200,  'age': 45, 'gender': 'M'},
    {'nombre': 'Atleta sub-3h',       'pr_21k_sec': 5100,  'age': 35, 'gender': 'M'},
    {'nombre': 'Atleta élite F',      'pr_21k_sec': 4200,  'age': 28, 'gender': 'F'},
]

print(f'{"Perfil":25s}  {"Dist":5s}  {"Ritmo (min/km)":20s}  {"Tiempo":12s}  {"Confianza":12s}')
print('─' * 82)

for perfil in perfiles:
    pr_key = {k: v for k, v in perfil.items() if k.startswith('pr_')}
    for dist in ['5K', '10K', '21K', '42K']:
        r = predict_race_time_range(pr_key, dist, age=perfil['age'], gender=perfil['gender'])
        if 'error' in r:
            continue
        nombre = perfil['nombre'] if dist == '5K' else ''
        print(f'{nombre:25s}  {dist:5s}  {r["pace_range_fmt"]:20s}  {r["time_range_fmt"]:12s}  {r["confidence"]:12s}')
    print()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ─── Figura 05_01: rango de MAE efectivo por distancia × confianza ───────────
# Muestra visualmente cómo el sistema declara más incertidumbre para distancias
# sin soporte empírico directo.

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle(
    'Sistema de predicción — Rango de ritmo para tres perfiles típicos\n'
    '(línea = ritmo central; banda = ±MAE efectivo del segmento)',
    fontsize=11
)

distances = ['5K', '10K', '21K', '42K']
dist_km   = [5.0, 10.0, 21.0975, 42.195]
colors    = ['#2196F3', '#4CAF50', '#E91E63']

for ax, perfil, color in zip(axes, perfiles, colors):
    pr_key = {k: v for k, v in perfil.items() if k.startswith('pr_')}
    centers, lows, highs, labels = [], [], [], []

    for dist in distances:
        r = predict_race_time_range(pr_key, dist, age=perfil['age'], gender=perfil['gender'])
        if 'error' in r:
            continue
        centers.append(r['pace_center_sec_km'] / 60)
        lows.append(r['pace_low_sec_km'] / 60)
        highs.append(r['pace_high_sec_km'] / 60)
        labels.append(dist)

    x = range(len(labels))
    ax.plot(x, centers, 'o-', color=color, lw=2, ms=7, label='Ritmo central')
    ax.fill_between(x, lows, highs, alpha=0.25, color=color, label='Rango ±MAE')
    ax.set_xticks(list(x))
    ax.set_xticklabels(labels)
    ax.set_ylabel('Ritmo (min/km)')
    ax.set_title(f'{perfil["nombre"]}\n(PR 21K={perfil["pr_21k_sec"]//60}:{perfil["pr_21k_sec"]%60:02d}, '
                 f'edad={perfil["age"]}, {perfil["gender"]})', fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda v, _: f'{int(v)}:{int((v % 1) * 60):02d}'
    ))
    ax.invert_yaxis()  # ritmos más rápidos arriba
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

plt.tight_layout()
fig_path = FIGS_DIR / '05_fig_01_rangos_ritmo_por_perfil.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura guardada: {fig_path.name}')

---
## 3. Análisis del rango de ritmo: qué lo determina y cómo se contrae

El ancho del rango depende de tres factores:
1. **Segmento del atleta:** corredores de 4h+ tienen mayor variabilidad inherente (MAE=12 min vs 2.5 min para elite)
2. **Distancia objetivo:** menos soporte empírico → mayor penalización → rango más ancho
3. **Capas activas:** agregar carga (Capa 3) y bienestar (Capa 4) reducirá el rango en versiones futuras

In [ ]:
# ─── Tabla de MAE efectivo por segmento × distancia ──────────────────────────
print('MAE efectivo (minutos) según segmento y distancia objetivo')
print('(determina el ancho de banda del rango de ritmo)')
print('=' * 60)
print(f'{"":10s}', end='')
for dist in ['5K', '10K', '21K', '42K']:
    print(f'{dist:>12s}', end='')
print()
print('-' * 60)

for segment, mae_base in CALIBRATED_MAE_MIN.items():
    if segment == 'default':
        continue
    print(f'{segment:10s}', end='')
    for dist in ['5K', '10K', '21K', '42K']:
        penalty = MAE_PENALTY_FACTOR[dist]
        mae_eff  = mae_base * penalty
        print(f'{mae_eff:12.1f}', end='')
    print()

print('=' * 60)
print('\nFactor de penalización por distancia (menor soporte empírico):')
for dist, factor in MAE_PENALTY_FACTOR.items():
    print(f'  {dist}: ×{factor}  ({EMPIRICAL_SUPPORT[dist][:45]}...)')

print()
print('─── CÓMO SE CONTRAE EL RANGO (expectativa con Capa 3) ─────────────────')
print('Capa 3 (carga: CTL/ATL/ACWR) → se espera reducir MAE ~10-20%')
print('Capa 4 (bienestar: check-in)  → ajuste fino ±5 min según fatiga/sueño')
print('Con todos los datos → rango más estrecho = predicción más útil para el atleta')

---
## 4. Papel de los PRs del formulario: señal más valiosa del sistema

Demostración de por qué el PR supera a la corrección demográfica sola.

In [ ]:
# ─── Comparación: con PR vs sin PR (solo demografía) ─────────────────────────
print('COMPARACIÓN: Con PR vs Sin PR — mismo atleta 42K')
print('=' * 65)

# Con PR de 21K
r_con_pr = predict_race_time_range({'pr_21k_sec': 5100}, '42K', age=35, gender='M')
print(f'\nCon PR (21K = 1:25:00):')
print(f'  Ritmo: {r_con_pr["pace_range_fmt"]}')
print(f'  Tiempo: {r_con_pr["time_range_fmt"]}')
print(f'  MAE: ±{r_con_pr["mae_effective_min"]} min')
print(f'  Confianza: {r_con_pr["confidence"]}')
print(f'  Capas: {r_con_pr["layers_active"]}')

# Sin PR
r_sin_pr = predict_race_time_range({}, '42K', age=35, gender='M')
print(f'\nSin PR (solo demografía):')
print(f'  Status: {r_sin_pr.get("error", "OK")}')
print(f'  Mensaje: {r_sin_pr.get("message", "")}')

print()
print('─── CONCLUSIÓN ─────────────────────────────────────────────────────────')
print(f'El PR reduce la incertidumbre de ~40-50 min (prior demográfico)')
print(f'  a ±{r_con_pr["mae_effective_min"]} min (Riegel + demografía).')
print(f'El PR es irreemplazable. Sin él, el sistema pide al atleta registrar')
print(f'al menos un tiempo de carrera para activar Capas 1+2.')

# ─── Demostración: el PR más cercano da mejor predicción ─────────────────────
print()
print('─── EFECTO DEL PR FUENTE EN LA PREDICCIÓN DE 42K ───────────────────────')
print('(mismo atleta, distintos PRs fuente — muestra degradación con distancia)')
prs_disponibles = [
    ('Solo 5K',  {'pr_5k_sec': 1320}),
    ('Solo 10K', {'pr_10k_sec': 2700}),
    ('Solo 21K', {'pr_21k_sec': 5100}),
    ('Solo 42K', {'pr_42k_sec': 10620}),
]
print(f'{"Fuente":10s}  {"Ritmo":22s}  {"MAE":8s}  {"Confianza"}')
for nombre, profile in prs_disponibles:
    r = predict_race_time_range(profile, '42K', age=35, gender='M')
    if 'error' in r:
        continue
    flag = '⬇️' if r['is_downward_extrapolation'] else '⬆️'
    print(f'{nombre:10s}  {r["pace_range_fmt"]:22s}  ±{r["mae_effective_min"]:5.1f}m  {r["confidence"]}  {flag}')

---
## 5. Evaluación de Run Club para la Capa 3

Run Club Marathon Performance Dataset tiene 80K filas con features de entrenamiento + `actual_finish_time_minutes`.
Verificamos si es usable como dataset real para entrenar la corrección por carga.

In [ ]:
from scipy import stats as scipy_stats

RC_PATH = DATA_DIR / 'Nuevo dataset Run Club Marathon Performance Dataset'
df_rc   = pd.read_csv(RC_PATH / 'train.csv')

print(f'Run Club train.csv: {df_rc.shape}')
print(f'Target: actual_finish_time_minutes')
print()

target = df_rc['actual_finish_time_minutes']
print('─── Distribución del target ────────────────────────────────────────')
print(f'  mean={target.mean():.1f} min  std={target.std():.1f} min')
print(f'  min={target.min():.0f}  p05={target.quantile(.05):.0f}  '
      f'p50={target.quantile(.50):.0f}  p95={target.quantile(.95):.0f}  max={target.max():.0f}')
print(f'  Valores únicos: {target.nunique()} (para {len(df_rc):,} filas)')

# Señal 1: ¿Los tiempos son enteros exactos? (real → muchos decimales)
pct_integers = (target == target.round()).mean()
print(f'  Fracción de valores enteros exactos: {pct_integers:.1%}')

# Señal 2: ¿La edad está uniformemente distribuida?
age_counts = df_rc['age'].value_counts()
age_cv = age_counts.std() / age_counts.mean()  # coef variación
print()
print('─── Distribución de edad ───────────────────────────────────────────')
print(f'  Valores únicos de edad: {df_rc["age"].nunique()}')
print(f'  Coef. de variación del conteo por edad: {age_cv:.3f}')
print(f'  (en datos reales: >0.30; en datos sintéticos: <0.05 indica uniforme)')

# Señal 3: Correlaciones con target
key_features = ['personal_best_minutes', 'vo2_max', 'weekly_mileage_km',
                'resting_heart_rate_bpm', 'age']
print()
print('─── Correlaciones con actual_finish_time_minutes ───────────────────')
corr = df_rc[key_features + ['actual_finish_time_minutes']].corr()['actual_finish_time_minutes']
for feat, val in corr.drop('actual_finish_time_minutes').items():
    comment = ''
    if abs(val) > 0.7:
        comment = '← muy alta (fórmula directa?)'
    elif abs(val) < 0.1:
        comment = '← muy baja (esperada >0.3 para esta feature)'
    print(f'  {feat:35s}: r={val:+.4f}  {comment}')

# Señal 4: Distribucion de weekly_mileage — real debería ser log-normal
print()
wm = df_rc['weekly_mileage_km']
stat, p_val = scipy_stats.kstest(wm, 'norm', args=(wm.mean(), wm.std()))
print('─── Test de normalidad en weekly_mileage_km ────────────────────────')
print(f'  KS test (vs normal): D={stat:.4f}, p={p_val:.4f}')
print(f'  (Datos reales de km semanales son log-normal con cola derecha larga)')

In [ ]:
# ─── Veredicto Run Club ───────────────────────────────────────────────────────
print('╔══════════════════════════════════════════════════════════════════════╗')
print('║         VEREDICTO: RUN CLUB PARA CAPA 3                            ║')
print('╠══════════════════════════════════════════════════════════════════════╣')
print('║                                                                      ║')
print('║  Señales de sinteticidad confirmadas:                               ║')
print('║  ❌ 97.5% de tiempos son enteros exactos (min exactos redondeados)   ║')
print('║  ❌ Edad uniformemente distribuida (CV < 0.03 — casi perfecta)       ║')
print('║  ❌ Solo 221 valores únicos de finish_time para 80K filas            ║')
print('║  ❌ personal_best r=0.856 con finish_time → generado por fórmula     ║')
print('║  ❌ weekly_mileage r=-0.11 con finish_time (esperado: r~-0.4)        ║')
print('║  ❌ vo2_max r=-0.22 (esperado en datos reales: r~-0.5 a -0.7)       ║')
print('║                                                                      ║')
print('║  DECISIÓN: DESCARTADO para entrenamiento de Capa 3.                 ║')
print('║  Cualquier modelo entrenado en este dataset no tiene validez         ║')
print('║  predictiva externa — aprendería la fórmula de generación,          ║')
print('║  no patrones reales de la fisiología del running.                   ║')
print('║                                                                      ║')
print('║  Puede usarse SOLO como: simulación, ilustración metodológica,      ║')
print('║  o validación de pipeline — nunca como datos de entrenamiento.      ║')
print('╚══════════════════════════════════════════════════════════════════════╝')

---
## 6. Estrategia para la Capa 3 con datos reales

Con Run Club descartado, las opciones reales son:

In [ ]:
capa3_opciones = pd.DataFrame([
    {
        'Opción':    '1. Injury Prediction (74 atletas)',
        'Fortaleza': 'Real, longitudinal, features CTL-like, 42K filas',
        'Limitación':'Target = lesión binaria (no tiempo de carrera)',
        'Uso':       'Objetivo secundario: validar acwr_zone() como clasificador'
    },
    {
        'Opción':    '2. Atleta propio (Strava + PRs reales)',
        'Fortaleza': 'Datos 100% reales, pipeline ya funcional',
        'Limitación':'n=1 — no generalizable estadísticamente',
        'Uso':       'Case study en tesis: demostrar Capa 3 con datos propios'
    },
    {
        'Opción':    '3. Injury dataset + Riegel cross-prediction',
        'Fortaleza': 'Features CTL/ATL/ACWR disponibles por semana por atleta',
        'Limitación':'No tiene tiempos de carrera como target directo',
        'Uso':       'Validar correlación carga → rendimiento relativo'
    },
    {
        'Opción':    '4. Recolectar datos en la app (largo plazo)',
        'Fortaleza': 'Dataset real + longitudinal + multi-atleta',
        'Limitación':'Requiere N atletas × K meses de uso activo',
        'Uso':       'Versión productiva de Capa 3 (fase futura)'
    },
])

print('Estrategias para Capa 3 (corrección por carga):')
print(capa3_opciones.to_string(index=False))

print()
print('─── RECOMENDACIÓN ──────────────────────────────────────────────────────')
print('Para la tesis: usar Opción 1 (lesión) + Opción 2 (case study atleta).')
print('  → La Opción 1 valida que el CTL/ATL/ACWR captura información real.')
print('  → La Opción 2 demuestra el sistema end-to-end con 1 atleta real.')
print('  → Ambos juntos forman un caso académico sólido y honesto.')
print('Para la app: implementar Capa 3 cuando haya datos de múltiples atletas.')

---
## 7. Interfaz de la función — qué expone al resto del sistema

In [ ]:
# ─── Demo de la salida completa (lo que ve la app y el endpoint FastAPI) ──────
import json

perfil_demo = {
    'pr_21k_sec': 5100,
    'pr_10k_sec': 2700,
}

resultado = predict_race_time_range(
    profile         = perfil_demo,
    target_distance = '42K',
    age             = 35,
    gender          = 'M',
)

print('Output completo de predict_race_time_range():')
print(json.dumps(resultado, indent=2, ensure_ascii=False))

In [ ]:
# ─── Figura 05_02: MAE efectivo por segmento × distancia ─────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(10, 5))

segments = ['elite', 'sub3h', '3to4h', '4hplus']
distances = ['5K', '10K', '21K', '42K']
seg_labels = {'elite': 'Élite (<2:30h)', 'sub3h': 'Sub-3h', '3to4h': '3–4h', '4hplus': '4h+'}
seg_colors = {'elite': '#1565C0', 'sub3h': '#2E7D32', '3to4h': '#F57F17', '4hplus': '#B71C1C'}

x = np.arange(len(distances))
width = 0.2

for i, seg in enumerate(segments):
    mae_vals = [
        CALIBRATED_MAE_MIN[seg] * MAE_PENALTY_FACTOR[d]
        for d in distances
    ]
    ax.bar(x + i * width - 0.3, mae_vals, width,
           label=seg_labels[seg], color=seg_colors[seg], alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(distances)
ax.set_ylabel('MAE efectivo (minutos)')
ax.set_xlabel('Distancia objetivo')
ax.set_title(
    'MAE efectivo del sistema por segmento de velocidad y distancia objetivo\n'
    '(incluye penalización por menor soporte empírico en distancias cortas)',
    fontsize=11
)
ax.legend(loc='upper left', fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.axvline(x=2.5, color='gray', ls='--', lw=1, alpha=0.5)
ax.text(2.7, ax.get_ylim()[1] * 0.9, 'Menor soporte\nempírico →', fontsize=8, color='gray')

plt.tight_layout()
fig_path = FIGS_DIR / '05_fig_02_mae_por_segmento_distancia.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura guardada: {fig_path.name}')

---
## 8. Síntesis: estado del sistema integrado

In [ ]:
print('╔══════════════════════════════════════════════════════════════════════════╗')
print('║    ESTADO DEL SISTEMA DE PREDICCIÓN MULTI-DISTANCIA — NB05            ║')
print('╠══════════════════════════════════════════════════════════════════════════╣')
print('║                                                                          ║')
print('║  ✅ Capa 0: verificación de consistencia de PRs — IMPLEMENTADA          ║')
print('║     • Detecta PRs contradictorios (Δ > 2×MAE_segmento)                 ║')
print('║     • Selecciona el PR fuente óptimo según distancia objetivo            ║')
print('║     • El PR de la misma distancia siempre es preferido                  ║')
print('║                                                                          ║')
print('║  ✅ Capa 1: Riegel calibrado por segmento — IMPLEMENTADA                ║')
print('║     • 4 segmentos: elite (1.0366), sub3h (1.0332), 3to4h (1.0613),     ║')
print('║       4h+ (1.1100)                                                      ║')
print('║     • MAE: 2.5–12 min según segmento (validado en Boston 103K)         ║')
print('║                                                                          ║')
print('║  ✅ Capa 2: corrección demográfica — IMPLEMENTADA (aproximada)          ║')
print('║     • Ajuste por edad y género basado en Boston 2015-2018               ║')
print('║     • Mejora ~1-2 min sobre Riegel solo para 42K                        ║')
print('║     • Para producción: serializar Ridge de NB04 y cargar aquí           ║')
print('║                                                                          ║')
print('║  ❌ Capa 3: corrección por carga — PENDIENTE                            ║')
print('║     • Run Club descartado: datos sintéticos confirmados                  ║')
print('║     • Estrategia: Injury dataset (objetivo secundario) + atleta propio  ║')
print('║                                                                          ║')
print('║  ❌ Capa 4: ajuste de bienestar — PENDIENTE (datos de la app)           ║')
print('║                                                                          ║')
print('║  Salida funcional:                                                       ║')
print('║     pace_range_fmt   → "5:11 – 5:29 min/km"                            ║')
print('║     time_range_fmt   → "3:38:50 – 3:52:10"                             ║')
print('║     confidence       → ALTA / MEDIA / MEDIA-BAJA / BAJA                ║')
print('║     layers_active    → ["Capa0", "Capa1", "Capa2"]                     ║')
print('║     empirical_support → declaración honesta por distancia               ║')
print('║                                                                          ║')
print('║  Módulo listo para integrar en FastAPI:                                  ║')
print('║     from src.ml.predictor import predict_race_time_range               ║')
print('║                                                                          ║')
print('╚══════════════════════════════════════════════════════════════════════════╝')

print()
print('Siguiente paso natural — Opción A (recomendada):')
print('  NB06: Capa 3 — Injury Prediction + case study atleta propio')
print('  Valida que CTL/ATL/ACWR captura información real de riesgo/rendimiento')
print()
print('Siguiente paso natural — Opción B (paralela, alto impacto en la app):')
print('  Integrar predict_race_time_range() en el endpoint FastAPI')
print('  GET /athletes/{cedula}/prediction?target=42K')